# Point of this Notebook

We will use these views later in the Dashboard and Genie Space

## View 1: today's signals with analyst-friendly flags

In [0]:
CREATE OR REPLACE VIEW raghavan_trading_signals.gold.v_daily_signals AS
SELECT s.symbol, s.trade_date, s.close_price, s.signal,
       s.rsi_14, s.macd_histogram, s.bb_position, s.volatility_20d, s.volume_ratio, s.scored_at,
       CASE WHEN s.rsi_14 < 30 THEN 'Oversold'
            WHEN s.rsi_14 > 70 THEN 'Overbought' ELSE 'Neutral' END AS rsi_zone,
       CASE WHEN s.volume_ratio > 2.0 THEN 'Unusual Volume'
            WHEN s.volume_ratio > 1.5 THEN 'Above Average' ELSE 'Normal' END AS volume_flag,
       CASE WHEN s.bb_position > 1.0 THEN 'Above Upper Band'
            WHEN s.bb_position < 0.0 THEN 'Below Lower Band' ELSE 'Within Bands' END AS bollinger_zone
FROM raghavan_trading_signals.gold.daily_signals s
WHERE s.trade_date = (SELECT MAX(trade_date) FROM raghavan_trading_signals.gold.daily_signals);

In [0]:
SELECT * FROM raghavan_trading_signals.gold.v_daily_signals

## View 2: signal performance (did the call come true the next day?)

In [0]:
CREATE OR REPLACE VIEW raghavan_trading_signals.gold.v_signal_performance AS
WITH x AS (
  SELECT symbol, trade_date, signal, predicted_direction, close_price,
         LEAD(close_price) OVER (PARTITION BY symbol ORDER BY trade_date) AS next_close,
         LEAD(close_price) OVER (PARTITION BY symbol ORDER BY trade_date) - close_price AS move
  FROM raghavan_trading_signals.gold.daily_signals)
SELECT symbol, trade_date, signal, close_price, next_close, move,
       CASE WHEN signal='BUY'  AND move > 0  THEN 'Correct'
            WHEN signal='SELL' AND move <= 0 THEN 'Correct' ELSE 'Incorrect' END AS outcome,
       ROUND(move / close_price * 100, 4) AS actual_return_pct
FROM x WHERE next_close IS NOT NULL;

In [0]:
SELECT * FROM raghavan_trading_signals.gold.v_signal_performance

## View 3: weekly accuracy summary

In [0]:
CREATE OR REPLACE VIEW raghavan_trading_signals.gold.v_accuracy_summary AS
SELECT DATE_TRUNC('week', trade_date) AS week,
       COUNT(*) AS total_signals,
       SUM(CASE WHEN outcome='Correct' THEN 1 ELSE 0 END) AS correct_signals,
       ROUND(SUM(CASE WHEN outcome='Correct' THEN 1 ELSE 0 END)*100.0/COUNT(*), 2) AS accuracy_pct,
       ROUND(AVG(actual_return_pct), 4) AS avg_return_pct,
       SUM(CASE WHEN signal='BUY'  THEN 1 ELSE 0 END) AS buy_count,
       SUM(CASE WHEN signal='SELL' THEN 1 ELSE 0 END) AS sell_count
FROM raghavan_trading_signals.gold.v_signal_performance
GROUP BY DATE_TRUNC('week', trade_date) ORDER BY week DESC;

In [0]:
SELECT * FROM raghavan_trading_signals.gold.v_accuracy_summary

## View 4: simulated strategy vs buy-and-hold (cumulative, compounded)

In [0]:
CREATE OR REPLACE VIEW raghavan_trading_signals.gold.v_portfolio_simulation AS
WITH daily AS (
  SELECT trade_date,
         ROUND(AVG(CASE WHEN signal='BUY' THEN actual_return_pct
                        WHEN signal='SELL' THEN -actual_return_pct ELSE 0 END), 4) AS strat,
         ROUND(AVG(actual_return_pct), 4) AS bench
  FROM raghavan_trading_signals.gold.v_signal_performance
  WHERE outcome IS NOT NULL GROUP BY trade_date)
SELECT trade_date, strat AS strategy_return_pct, bench AS benchmark_return_pct,
       strat - bench AS alpha_pct,
       ROUND(EXP(SUM(LN(1+strat/100)) OVER (ORDER BY trade_date))*100-100, 4) AS cumulative_strategy_pct,
       ROUND(EXP(SUM(LN(1+bench/100)) OVER (ORDER BY trade_date))*100-100, 4) AS cumulative_benchmark_pct
FROM daily ORDER BY trade_date;

In [0]:
SELECT * FROM raghavan_trading_signals.gold.v_portfolio_simulation